# einops-rearrange-flatten — worked example 3: Flatten batch and time axes into a single sequence axis for sequence modeling

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange-flatten`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Some sequence models process a flat batch of frames rather than a batch of sequences. The pattern `'b t d -> (b t) d'` merges batch and time into a single leading axis, producing shape `(B*T, D)`. The model processes all `B*T` frames independently. After processing, the inverse `'(b t) d -> b t d'` restores the batch structure.

## Worked solution

Input: video frames of shape `(B=2, T=8, D=128)` — 2 videos, 8 frames each, 128-dim embedding.

**Pattern:** `'b t d -> (b t) d'`.

The `(b t)` group on the right merges `b` and `t` into a single axis of size `2*8=16`. The `d` axis is untouched.

**Result shape:** `(16, 128)` — 16 frames, each with 128 features. A batch-agnostic linear layer can now process all frames without caring about which video they came from.

**Inverse:** `rearrange(out, '(b t) d -> b t d', b=B)` restores the `(B, T, D)` structure for temporal aggregation.

In [ ]:
import torch as t
from einops import rearrange

t.manual_seed(5)
B, T, D = 3, 6, 64
frames = t.randn(B, T, D)

def pack_batch_time(x):
    return rearrange(x, 'b t d -> (b t) d')

def unpack_batch_time(flat, B):
    return rearrange(flat, '(b t) d -> b t d', b=B)

packed = pack_batch_time(frames)
print('Frames shape:', frames.shape)   # (3, 6, 64)
print('Packed shape:', packed.shape)   # (18, 64)
assert packed.shape == (B * T, D)

# Simulate a pointwise linear pass
processed = packed * 2.0  # placeholder operation

unpacked = unpack_batch_time(processed, B)
assert unpacked.shape == (B, T, D)
print('Unpacked shape:', unpacked.shape)
assert t.allclose(unpacked, frames * 2.0)
print('Round-trip correct:', True)